## Entrega práctica 3 - parte 2
Pamela Huacca Arce - Paula Andrea C. Cano

In [2]:
pip install seaborn

  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
Using cached seaborn-0.13.2-py3-none-any.whl (294 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import mne
import numpy as np
import pandas as pd
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import os

Para la solución de la práctica, extrajimos los datos de los sujetos impares entre el 15 y el 33. De este grupo utilizamos los datos etiquetados como run 4, 8 y 12, ya que estos correspondían a las tareas de imaginería, clasificadas dentro de estas run como TASK2T1 y TASK2T2, siendo T1 y T2 indicadores para la lateralidad izquiera y derecha, respectivemante.

A estos datos aplicamos un filtro de paso de banda queriendo conservar solo las frecuencias de interes y eliminar ruidos por fuera de estas.

Como resumen, no se hallaron diferencias estadísticas significativas. Se realizó un análisis en distintas ventanas de tiempo y a diferencias frecuencias de corte, para descartar que se debiera a esto, pero en ninguno de los distintos análisis de logró hallarlas. Creemos que esto puede deberse a que o basta solo son un filtro de paso de banda para limpiar la señal, por lo que se espera que una vez avanzados en el tema de filtros en el curso, podamos aplicarlo y buscar diferencias significativas. Por ahora, se dejan creadas las funciones, y adicionalmente se crea un data frame en el que se alamacenan los p-valores obtenidos, esto para mostrar los valores obtenidos.




In [35]:
def procesar_sujeto(ruta_archivo, tmax):
    """
    Ralizamos un filtro de paso de banda entre 8 Hz y 30 Hz, ya que encontramos en la literatura
    que las bandas Alfa y Beta son relevantes para la tarea de imaginación motora [1].
    """
    raw = mne.io.read_raw_eeglab(ruta_archivo, preload=True, verbose=False)
    raw.pick(['eeg']) 
    
    raw.filter(l_freq=8.0, h_freq=30.0, verbose=False)
    
    eventos, id_eventos = mne.events_from_annotations(raw, verbose=False)
    epocas = mne.Epochs(raw, eventos, event_id=id_eventos, tmin=0, tmax=tmax, 
                        baseline=None, preload=True, verbose=False)
    
    # TASK2T1 = Imaginar Mano Izquierda | TASK2T2 = Imaginar Mano Derecha
    data_izquierda = epocas['TASK2T1'][:8].get_data() 
    data_derecha = epocas['TASK2T2'][:8].get_data()
    
    nombres_canales = raw.ch_names[:64] # Extraemos los nombres de los canales
    
    return data_izquierda, data_derecha, nombres_canales

    # [1] https://www.redalyc.org/pdf/342/34247483006.pdf

In [ ]:
# PUNTO 1 - CÁLCULO DE LA AMPLITUD EFECTIVA (RMS) PARA CADA CANAL Y SUJETO

def calcular_rms_promedio(epocas_data):
    """
    Calcula el RMS temporal y promedia las épocas.
    Entrada: (8, 64, N) -> Salida: array de 64 valores.
    """
    # RMS sobre el tiempo (eje 2)
    rms_por_epoca = np.sqrt(np.mean(epocas_data**2, axis=2)) 
    # Promedio sobre las épocas (eje 0)
    rms_promedio = np.mean(rms_por_epoca, axis=0)
    return rms_promedio

In [51]:
# PUNTO 2 - CONSTRUCCIÓN DE LA BASE DE DATOS POBLACIONAL

rutas_sujetos = [
    r"D:\LabBiosenales\data_lab3\participants\sub-015_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-017_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-019_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-021_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-023_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-025_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-027_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-029_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-031_task-motion_run-8_eeg.set",
    r"D:\LabBiosenales\data_lab3\participants\sub-033_task-motion_run-8_eeg.set"
]

datos_izquierda = []
datos_derecha = []
nombres_canales_global = None

print("Iniciando procesamiento de sujetos...")

for i, ruta in enumerate(rutas_sujetos):
    print(f"  Procesando Sujeto {i+1}...")
    try:
        # Extraer matrices
        data_izq, data_der, canales = procesar_sujeto(ruta, tmax=4.0)
        
        # Guardar los nombres de los canales en la primera iteración
        if nombres_canales_global is None:
            nombres_canales_global = canales
            
        rms_izq = calcular_rms_promedio(data_izq)
        rms_der = calcular_rms_promedio(data_der)
        
        datos_izquierda.append(rms_izq)
        datos_derecha.append(rms_der)
    except Exception as e:
        print(f"  Error con el sujeto {i+1}: {e}")

df_izquierda = pd.DataFrame(datos_izquierda, columns=nombres_canales_global)
df_derecha = pd.DataFrame(datos_derecha, columns=nombres_canales_global)

Iniciando procesamiento de sujetos...
  Procesando Sujeto 1...
  Procesando Sujeto 2...
  Procesando Sujeto 3...
  Procesando Sujeto 4...
  Procesando Sujeto 5...
  Procesando Sujeto 6...
  Procesando Sujeto 7...
  Procesando Sujeto 8...
  Procesando Sujeto 9...
  Procesando Sujeto 10...


In [ ]:
# PUNTO 3 - IDENTIFICACIÓN DE CANALES DIFERENCIALES MEDIANTE ANÁLISIS ESTADÍSTICO

print("\nIniciando análisis estadístico...")
canales_significativos = {}
resultados_todos_canales = []
for canal in df_izquierda.columns:
    grupo_izq = df_izquierda[canal]
    grupo_der = df_derecha[canal]
    
    # Pruebas de supuestos
    _, p_shapiro_izq = stats.shapiro(grupo_izq)
    _, p_shapiro_der = stats.shapiro(grupo_der)
    normalidad = (p_shapiro_izq > 0.05) and (p_shapiro_der > 0.05)
    
    _, p_levene = stats.levene(grupo_izq, grupo_der)
    homocedasticidad = (p_levene > 0.05)
    
    # Prueba de hipótesis
    if normalidad and homocedasticidad:
        _, p_valor = stats.ttest_ind(grupo_izq, grupo_der)
        prueba = "T-Student"
    else:
        _, p_valor = stats.mannwhitneyu(grupo_izq, grupo_der)
        prueba = "Mann-Whitney"

    resultados_todos_canales.append({
        'Canal': canal,
        'p-valor': p_valor,
        'Prueba Usada': prueba })

    if p_valor < 0.05:
        canales_significativos[canal] = p_valor
    
df_resultados = pd.DataFrame(resultados_todos_canales)
df_resultados = df_resultados.sort_values(by='p-valor', ascending=True)
df_resultados = df_resultados.reset_index(drop=True)

print("\nCanales con diferencias significativas (p < 0.05):")
for c, p in canales_significativos.items():
    print(f"Canal {c}: p-valor = {p:.4f}")


Iniciando análisis estadístico...

Canales con diferencias significativas (p < 0.05):


In [ ]:
"""
Creamos un Data frame de resultados estadísticos para todos los canales ordenados por p-valor
de froma ascendente.
Notamos que el p-valor menor es 0.623, lo que supera por mucho el umbral de 0.05,
indicando que no hay diferencias estadísticamente significativas entre las condiciones
de imaginación motora para ningún canal.
"""

df_resultados

,Canal,p-valor,Prueba Usada
0,C5,0.623176,Mann-Whitney
1,Cp3,0.775506,T-Student
2,P3,0.783318,T-Student
3,Fpz,0.791337,Mann-Whitney
4,C3,0.797935,T-Student
...,...,...,...
59,F5,0.996400,T-Student
60,P2,0.996871,T-Student
61,Poz,0.997021,T-Student
62,Iz,0.998877,T-Student


In [ ]:
# VISUALIZACIÓN (BOXPLOT)

if canales_significativos: # El canal con el p-valor más bajo
    mejor_canal = min(canales_significativos, key=canales_significativos.get)
    
    datos_grafico = pd.DataFrame({
        'RMS': pd.concat([df_izquierda[mejor_canal], df_derecha[mejor_canal]]),
        'Intención': ['Mano Izquierda']*len(df_izquierda) + ['Mano Derecha']*len(df_derecha)})

    plt.figure(figsize=(8, 6))
    sns.boxplot(x='Intención', y='RMS', data=datos_grafico, palette='Set2')
    sns.stripplot(x='Intención', y='RMS', data=datos_grafico, color='black', alpha=0.5, jitter=True)
    plt.title(f'Distribución de Energía (RMS) en el Canal {mejor_canal}')
    plt.ylabel('Amplitud Efectiva (RMS)')
    plt.xlabel('Tarea de Imaginería Motora')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.show()
else:
    print("No se encontraron canales con diferencias significativas para graficar.")

No se encontraron canales con diferencias significativas para graficar.
